# Capstone Assessment: Heart Disease Classification with Decision Trees

Every dataset so far has been synthetic, built so the "right" pattern was clean and the numbers behaved. This
notebook uses a real one instead: the **Heart Failure Prediction Dataset**
([fedesoriano, Kaggle](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction)) — 918 patient
records with 11 clinical measurements and a binary outcome, `HeartDisease` (1 = heart disease, 0 = normal). It's
saved locally as `../data/heart_failure.csv`, so you don't need a Kaggle account or API key to run this.

This is a capstone, not a new lesson: every technique below — EDA, encoding, `train_test_split`, fitting a
`DecisionTreeClassifier`, reading `max_depth` trade-offs, `export_text`, feature importance — was covered in
Sessions 1-5. The exercises follow the same `### ✏️ Try it yourself` → `TODO` → collapsed solution format as the
other workbooks, but real data is messier than synthetic data, so a couple of exercises ask you to notice
something is *wrong* before fixing it. Nothing here is graded, but it's meant to check whether the ideas
actually stuck.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

pd.set_option("display.precision", 3)

## 1. Meet the Dataset

### ✏️ Try it yourself

Load `../data/heart_failure.csv` into `heart_df`. Print its shape, its `.dtypes`, and how many missing values
each column has (`.isna().sum()`).

In [2]:
# TODO: load heart_df from ../data/heart_failure.csv, then print its shape, dtypes, and isna().sum()

<details>
<summary>Show solution</summary>

```python
heart_df = pd.read_csv("../data/heart_failure.csv")
print(heart_df.shape)
print()
print(heart_df.dtypes)
print()
print(heart_df.isna().sum())
```

</details>

Continuing with the loaded data:

In [3]:
heart_df = pd.read_csv("../data/heart_failure.csv")
print(heart_df.shape)
print()
print(heart_df.dtypes)
print()
print(heart_df.isna().sum())

(918, 12)

Age                 int64
Sex                object
ChestPainType      object
RestingBP           int64
Cholesterol         int64
FastingBS           int64
RestingECG         object
MaxHR               int64
ExerciseAngina     object
Oldpeak           float64
ST_Slope           object
HeartDisease        int64
dtype: object

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64


> ### 💭 Think about it
>
> `isna().sum()` reports zero missing values in every column. Do you trust that completely -- for *every*
> column, including the numeric ones? What would a missing measurement look like in a column that only accepts
> numbers?

## 2. Exploratory Data Analysis

### ✏️ Try it yourself

Count how many rows have `Cholesterol == 0` and how many have `RestingBP == 0`.

In [4]:
# TODO: print the number of rows where Cholesterol == 0, and the number where RestingBP == 0

<details>
<summary>Show solution</summary>

```python
print("Cholesterol == 0:", (heart_df["Cholesterol"] == 0).sum())
print("RestingBP == 0:  ", (heart_df["RestingBP"] == 0).sum())
```

</details>

Continuing with the worked counts:

In [5]:
print("Cholesterol == 0:", (heart_df["Cholesterol"] == 0).sum())
print("RestingBP == 0:  ", (heart_df["RestingBP"] == 0).sum())

Cholesterol == 0: 172
RestingBP == 0:   1


> ### 🧑‍🏫 Instructor note
>
> A living patient doesn't have 0 mg/dL cholesterol. This is a classic real-world pattern: a missing measurement
> gets encoded as 0 instead of `NaN`, so `.isna()` finds nothing wrong. Nearly a fifth of the `Cholesterol`
> column is silently missing this way. Always sanity-check the *plausible range* of a numeric column, not just
> whether pandas thinks it's missing.

### ✏️ Try it yourself

Fix it: in `heart_df`, replace `0` with `NaN` in both `Cholesterol` and `RestingBP`, then fill the resulting
`NaN`s in each column with that column's median.

In [6]:
# TODO: replace 0 with NaN in Cholesterol and RestingBP, then fill each column's NaNs with its own median

<details>
<summary>Show solution</summary>

```python
heart_df["Cholesterol"] = heart_df["Cholesterol"].replace(0, np.nan)
heart_df["RestingBP"] = heart_df["RestingBP"].replace(0, np.nan)

heart_df["Cholesterol"] = heart_df["Cholesterol"].fillna(heart_df["Cholesterol"].median())
heart_df["RestingBP"] = heart_df["RestingBP"].fillna(heart_df["RestingBP"].median())

print("Cholesterol == 0 after fix:", (heart_df["Cholesterol"] == 0).sum())
print("RestingBP == 0 after fix:  ", (heart_df["RestingBP"] == 0).sum())
```

</details>

Continuing with the fixed columns -- every later exercise uses this cleaned `heart_df`:

In [7]:
heart_df["Cholesterol"] = heart_df["Cholesterol"].replace(0, np.nan)
heart_df["RestingBP"] = heart_df["RestingBP"].replace(0, np.nan)

heart_df["Cholesterol"] = heart_df["Cholesterol"].fillna(heart_df["Cholesterol"].median())
heart_df["RestingBP"] = heart_df["RestingBP"].fillna(heart_df["RestingBP"].median())

print("Cholesterol == 0 after fix:", (heart_df["Cholesterol"] == 0).sum())
print("RestingBP == 0 after fix:  ", (heart_df["RestingBP"] == 0).sum())

Cholesterol == 0 after fix: 0
RestingBP == 0 after fix:   0


### ✏️ Try it yourself

Compute the class balance of the target: `heart_df["HeartDisease"].value_counts(normalize=True)`.

In [8]:
# TODO: compute and print the normalized value_counts of HeartDisease

<details>
<summary>Show solution</summary>

```python
heart_df["HeartDisease"].value_counts(normalize=True)
```

</details>

Continuing with the worked result, plotted:

<details>
<summary>Show code</summary>

```python
class_counts = heart_df["HeartDisease"].value_counts().sort_index()

class_balance_figure = go.Figure()
class_balance_figure.add_trace(
    go.Bar(x=["No disease (0)", "Disease (1)"], y=class_counts.values)
)
class_balance_figure.update_layout(
    title="Class balance: HeartDisease",
    yaxis_title="Number of patients",
    template="plotly_white",
)
class_balance_figure.show()
```

</details>

In [9]:
class_counts = heart_df["HeartDisease"].value_counts().sort_index()

class_balance_figure = go.Figure()
class_balance_figure.add_trace(
    go.Bar(x=["No disease (0)", "Disease (1)"], y=class_counts.values)
)
class_balance_figure.update_layout(
    title="Class balance: HeartDisease",
    yaxis_title="Number of patients",
    template="plotly_white",
)
class_balance_figure.show()

Not perfectly balanced, but close enough that plain accuracy is still a reasonable headline number -- unlike a
99%-vs-1% split, where a model that always predicts the majority class would look great on accuracy alone.

## A Categorical Feature, By Eye

Before touching the numeric columns, a quick look at one categorical one: does chest pain type actually relate
to the outcome?

<details>
<summary>Show code</summary>

```python
pain_by_outcome = pd.crosstab(heart_df["ChestPainType"], heart_df["HeartDisease"], normalize="index")

pain_figure = go.Figure()
pain_figure.add_trace(go.Bar(name="No disease", x=pain_by_outcome.index, y=pain_by_outcome[0]))
pain_figure.add_trace(go.Bar(name="Disease", x=pain_by_outcome.index, y=pain_by_outcome[1]))
pain_figure.update_layout(
    title="HeartDisease rate by ChestPainType",
    yaxis_title="Share of patients",
    barmode="stack",
    template="plotly_white",
)
pain_figure.show()
```

</details>

In [10]:
pain_by_outcome = pd.crosstab(heart_df["ChestPainType"], heart_df["HeartDisease"], normalize="index")

pain_figure = go.Figure()
pain_figure.add_trace(go.Bar(name="No disease", x=pain_by_outcome.index, y=pain_by_outcome[0]))
pain_figure.add_trace(go.Bar(name="Disease", x=pain_by_outcome.index, y=pain_by_outcome[1]))
pain_figure.update_layout(
    title="HeartDisease rate by ChestPainType",
    yaxis_title="Share of patients",
    barmode="stack",
    template="plotly_white",
)
pain_figure.show()

`ASY` (asymptomatic) stands out sharply -- worth remembering when feature importance comes up later.

### ✏️ Try it yourself

Compute the correlation matrix of the numeric columns: `Age`, `RestingBP`, `Cholesterol`, `FastingBS`, `MaxHR`,
`Oldpeak`.

In [11]:
numeric_cols = ["Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak"]
# TODO: compute heart_df[numeric_cols].corr()

<details>
<summary>Show solution</summary>

```python
numeric_cols = ["Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak"]
correlation_matrix = heart_df[numeric_cols].corr()
correlation_matrix
```

</details>

Continuing with the worked matrix, plotted:

<details>
<summary>Show code</summary>

```python
correlation_matrix = heart_df[numeric_cols].corr()

corr_figure = go.Figure()
corr_figure.add_trace(
    go.Heatmap(
        z=correlation_matrix.values,
        x=correlation_matrix.columns,
        y=correlation_matrix.columns,
        colorscale="RdBu",
        zmid=0,
    )
)
corr_figure.update_layout(title="Correlation between numeric features", template="plotly_white")
corr_figure.show()
```

</details>

In [12]:
correlation_matrix = heart_df[numeric_cols].corr()

corr_figure = go.Figure()
corr_figure.add_trace(
    go.Heatmap(
        z=correlation_matrix.values,
        x=correlation_matrix.columns,
        y=correlation_matrix.columns,
        colorscale="RdBu",
        zmid=0,
    )
)
corr_figure.update_layout(title="Correlation between numeric features", template="plotly_white")
corr_figure.show()

> ### 💭 Think about it
>
> None of the numeric features are strongly correlated with each other. Does that tell you anything about
> whether they're individually related to `HeartDisease`? Why or why not?

## 3. Preprocessing

`FastingBS` is already `0`/`1`, so it doesn't need encoding even though it represents a category (fasting blood
sugar > 120 mg/dL or not). The five text columns -- `Sex`, `ChestPainType`, `RestingECG`, `ExerciseAngina`,
`ST_Slope` -- do.

### ✏️ Try it yourself

Build `X` by one-hot encoding those five columns with `pd.get_dummies(..., drop_first=True)` on `heart_df`
(dropping `HeartDisease`), and build `y` as `heart_df["HeartDisease"]`.

In [13]:
categorical_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
# TODO: build X (one-hot encoded, HeartDisease dropped) and y (HeartDisease)

<details>
<summary>Show solution</summary>

```python
categorical_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
X = pd.get_dummies(heart_df.drop(columns="HeartDisease"), columns=categorical_cols, drop_first=True)
y = heart_df["HeartDisease"]
print(X.shape)
X.head()
```

</details>

Continuing with the encoded features:

In [14]:
categorical_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
X = pd.get_dummies(heart_df.drop(columns="HeartDisease"), columns=categorical_cols, drop_first=True)
y = heart_df["HeartDisease"]
print(X.shape)
X.head()

(918, 15)


,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,40,140.0,289.0,0,172,0.0,True,True,False,False,True,False,False,False,True
1,49,160.0,180.0,0,156,1.0,False,False,True,False,True,False,False,True,False
2,37,130.0,283.0,0,98,0.0,True,True,False,False,False,True,False,False,True
3,48,138.0,214.0,0,108,1.5,False,False,False,False,True,False,True,True,False
4,54,150.0,195.0,0,122,0.0,True,False,True,False,True,False,False,False,True


### ✏️ Try it yourself

Split `X, y` into `X_train, X_test, y_train, y_test` using `train_test_split` with `test_size=0.3,
random_state=0, stratify=y` -- same settings every earlier session used.

In [15]:
# TODO: split X, y into X_train, X_test, y_train, y_test

<details>
<summary>Show solution</summary>

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print(f"Training rows: {len(X_train)}")
print(f"Test rows:     {len(X_test)}")
```

</details>

Continuing with the split:

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print(f"Training rows: {len(X_train)}")
print(f"Test rows:     {len(X_test)}")

Training rows: 642
Test rows:     276


## 4. Fitting and Evaluating a Decision Tree

### ✏️ Try it yourself

Fit a `DecisionTreeClassifier(max_depth=4, random_state=0)` on `X_train, y_train`, and print its train and test
accuracy.

In [17]:
# TODO: fit heart_tree_model and print its train/test accuracy

<details>
<summary>Show solution</summary>

```python
heart_tree_model = DecisionTreeClassifier(max_depth=4, random_state=0)
heart_tree_model.fit(X_train, y_train)
print(f"Train accuracy: {heart_tree_model.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {heart_tree_model.score(X_test, y_test):.3f}")
```

</details>

Continuing with the worked model:

In [18]:
heart_tree_model = DecisionTreeClassifier(max_depth=4, random_state=0)
heart_tree_model.fit(X_train, y_train)
print(f"Train accuracy: {heart_tree_model.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {heart_tree_model.score(X_test, y_test):.3f}")

Train accuracy: 0.868
Test accuracy:  0.844


## Beyond Accuracy: Confusion Matrix and Classification Report

Session 3 hand-built a confusion matrix from scratch, to see exactly what it's counting. Now that you know what
it means, `sklearn.metrics` gives you that table -- plus **precision** (of everything predicted positive, how
much actually was), **recall** (of everything actually positive, how much was caught), and **f1-score** (their
balance) -- in two function calls.

### ✏️ Try it yourself

Predict on `X_test`, then build a confusion matrix with `confusion_matrix(y_test, y_pred)` and print
`classification_report(y_test, y_pred)`.

In [19]:
# TODO: predict y_pred on X_test, then print a confusion matrix and a classification report

<details>
<summary>Show solution</summary>

```python
y_pred = heart_tree_model.predict(X_test)

confusion_table = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["Actual: no disease", "Actual: disease"],
    columns=["Predicted: no disease", "Predicted: disease"],
)
print(confusion_table)
print()
print(classification_report(y_test, y_pred))
```

</details>

Continuing with the worked evaluation:

In [20]:
y_pred = heart_tree_model.predict(X_test)

confusion_table = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["Actual: no disease", "Actual: disease"],
    columns=["Predicted: no disease", "Predicted: disease"],
)
print(confusion_table)
print()
print(classification_report(y_test, y_pred))

                    Predicted: no disease  Predicted: disease
Actual: no disease                    106                  17
Actual: disease                        26                 127

              precision    recall  f1-score   support

           0       0.80      0.86      0.83       123
           1       0.88      0.83      0.86       153

    accuracy                           0.84       276
   macro avg       0.84      0.85      0.84       276
weighted avg       0.85      0.84      0.84       276



> ### 🙋 Ask the class
>
> For a screening tool like this one, would you rather reduce false positives (predicting disease when there
> isn't one) or false negatives (predicting no disease when there is one)? What does that preference imply
> about which metric -- precision or recall -- matters more here?

### ✏️ Try it yourself

For `max_depth` in `[1, 2, 3, 4, 6, None]`: fit a `DecisionTreeClassifier`, and record its train accuracy and
test accuracy. Put the results in a DataFrame.

In [21]:
depth_results = []
for depth in [1, 2, 3, 4, 6, None]:
    # TODO: fit a DecisionTreeClassifier(max_depth=depth, random_state=0), then append a dict with
    #       "max_depth", "train_accuracy", "test_accuracy" to depth_results
    pass

pd.DataFrame(depth_results)

""


<details>
<summary>Show solution</summary>

```python
depth_results = []
for depth in [1, 2, 3, 4, 6, None]:
    depth_model = DecisionTreeClassifier(max_depth=depth, random_state=0)
    depth_model.fit(X_train, y_train)
    depth_results.append({
        "max_depth": depth if depth is not None else "None (unlimited)",
        "train_accuracy": depth_model.score(X_train, y_train),
        "test_accuracy": depth_model.score(X_test, y_test),
    })

pd.DataFrame(depth_results)
```

</details>

Continuing with the full sweep so we can plot it:

<details>
<summary>Show code</summary>

```python
depth_labels = ["1", "2", "3", "4", "6", "None"]
train_accuracies = []
test_accuracies = []
for depth in [1, 2, 3, 4, 6, None]:
    depth_model = DecisionTreeClassifier(max_depth=depth, random_state=0)
    depth_model.fit(X_train, y_train)
    train_accuracies.append(depth_model.score(X_train, y_train))
    test_accuracies.append(depth_model.score(X_test, y_test))

depth_figure = go.Figure()
depth_figure.add_trace(go.Scatter(x=depth_labels, y=train_accuracies, mode="lines+markers", name="Train accuracy"))
depth_figure.add_trace(go.Scatter(x=depth_labels, y=test_accuracies, mode="lines+markers", name="Test accuracy"))
depth_figure.update_layout(
    title="Accuracy vs. max_depth",
    xaxis_title="max_depth",
    yaxis_title="Accuracy",
    template="plotly_white",
)
depth_figure.show()
```

</details>

In [22]:
depth_labels = ["1", "2", "3", "4", "6", "None"]
train_accuracies = []
test_accuracies = []
for depth in [1, 2, 3, 4, 6, None]:
    depth_model = DecisionTreeClassifier(max_depth=depth, random_state=0)
    depth_model.fit(X_train, y_train)
    train_accuracies.append(depth_model.score(X_train, y_train))
    test_accuracies.append(depth_model.score(X_test, y_test))

depth_figure = go.Figure()
depth_figure.add_trace(go.Scatter(x=depth_labels, y=train_accuracies, mode="lines+markers", name="Train accuracy"))
depth_figure.add_trace(go.Scatter(x=depth_labels, y=test_accuracies, mode="lines+markers", name="Test accuracy"))
depth_figure.update_layout(
    title="Accuracy vs. max_depth",
    xaxis_title="max_depth",
    yaxis_title="Accuracy",
    template="plotly_white",
)
depth_figure.show()

> ### 🧑‍🏫 Instructor note
>
> Same shape as Session 4's synthetic data: train accuracy keeps climbing toward 1.000 as depth grows, test
> accuracy peaks somewhere in the middle and then flattens or drops. On real data the peak is noisier and less
> dramatic than on the synthetic dataset -- that's expected, not a bug.

## Reading the Tree

```python
depth3_model = DecisionTreeClassifier(max_depth=3, random_state=0)
depth3_model.fit(X_train, y_train)
print(export_text(depth3_model, feature_names=list(X.columns)))
```

In [23]:
depth3_model = DecisionTreeClassifier(max_depth=3, random_state=0)
depth3_model.fit(X_train, y_train)
print(export_text(depth3_model, feature_names=list(X.columns)))

|--- ST_Slope_Up <= 0.50
|   |--- MaxHR <= 150.50
|   |   |--- Sex_M <= 0.50
|   |   |   |--- class: 1
|   |   |--- Sex_M >  0.50
|   |   |   |--- class: 1
|   |--- MaxHR >  150.50
|   |   |--- ChestPainType_NAP <= 0.50
|   |   |   |--- class: 1
|   |   |--- ChestPainType_NAP >  0.50
|   |   |   |--- class: 0
|--- ST_Slope_Up >  0.50
|   |--- ExerciseAngina_Y <= 0.50
|   |   |--- Oldpeak <= 2.25
|   |   |   |--- class: 0
|   |   |--- Oldpeak >  2.25
|   |   |   |--- class: 1
|   |--- ExerciseAngina_Y >  0.50
|   |   |--- ChestPainType_ATA <= 0.50
|   |   |   |--- class: 1
|   |   |--- ChestPainType_ATA >  0.50
|   |   |   |--- class: 0



Longer feature names than Session 4's `hours_studied`/`practice_problems` -- one-hot encoding turns a single
categorical column into several `column_value` features, and the tree can split on any of them individually.

### ✏️ Try it yourself

Using `heart_tree_model` (the `max_depth=4` tree from earlier), build a `pd.Series` of
`.feature_importances_` indexed by `X.columns`, sort it descending, and make a horizontal bar chart of the top
8.

In [24]:
# TODO: build importance_series from heart_tree_model.feature_importances_, sorted descending,
#       then plot the top 8 as a horizontal bar chart

<details>
<summary>Show solution</summary>

```python
importance_series = pd.Series(heart_tree_model.feature_importances_, index=X.columns).sort_values(ascending=False)
top_importances = importance_series.head(8)

importance_figure = go.Figure()
importance_figure.add_trace(
    go.Bar(x=top_importances.values[::-1], y=top_importances.index[::-1], orientation="h")
)
importance_figure.update_layout(
    title="Top 8 feature importances",
    xaxis_title="Importance",
    template="plotly_white",
)
importance_figure.show()
```

</details>

Continuing with the worked importances:

In [25]:
importance_series = pd.Series(heart_tree_model.feature_importances_, index=X.columns).sort_values(ascending=False)
top_importances = importance_series.head(8)

importance_figure = go.Figure()
importance_figure.add_trace(
    go.Bar(x=top_importances.values[::-1], y=top_importances.index[::-1], orientation="h")
)
importance_figure.update_layout(
    title="Top 8 feature importances",
    xaxis_title="Importance",
    template="plotly_white",
)
importance_figure.show()

`ST_Slope_Up` dominates the list, with `ExerciseAngina_Y` and `MaxHR` following. `ChestPainType` doesn't show up
as strongly here even though the crosstab above made it look important by eye -- two things are going on:
`ST_Slope` alone already captures much of the same signal, so the tree has less need to also split on chest
pain type, and `ChestPainType_ASY` (the category that stood out most in the crosstab) isn't even a column here
-- `drop_first=True` dropped it as the baseline category, so its effect is folded into what every *other*
`ChestPainType_*` column is being compared against rather than getting its own importance score.

## What We Covered Today

- Real datasets hide missing data in ways `.isna()` won't catch -- placeholder values like `0` need a plausible-range check, not just a null check.
- One-hot encoding (`pd.get_dummies`) turns text categories into numeric columns a tree can split on; already-numeric binary flags (like `FastingBS`) don't need it.
- `confusion_matrix` and `classification_report` give you precision/recall/f1 in two calls, instead of hand-building the table like Session 3 did.
- The train-vs-test accuracy gap across `max_depth` values shows overfitting on real data the same way it did on synthetic data, just noisier.
- Feature importance on real data lines up with patterns you can often already see by eye in the EDA.

## Final Check

1. `heart_df.isna().sum()` reported zero missing values, yet a fifth of `Cholesterol` was effectively missing. What general lesson does that teach about trusting a "no missing values" result?
2. In this dataset, what does a **false negative** mean in real terms, and why might it matter more than a false positive for a screening tool like this one?
3. The `max_depth` sweep showed test accuracy rise, peak, then flatten or fall as depth increased, while train accuracy kept climbing toward 1.000. Explain what's happening in terms of overfitting.
4. A classmate says "`Cholesterol` has low feature importance in our tree, so cholesterol must not matter medically for heart disease." What's wrong with that reasoning?

<details>
<summary>Show solution</summary>

1. A "no missing values" check only catches actual `NaN`/null entries. Data can still be functionally missing
   if it was encoded as a placeholder value (here, `0`) instead of a proper null. Always sanity-check a numeric
   column's plausible range, not just whether pandas flags it as missing.
2. A false negative here means predicting "no disease" for a patient who actually has heart disease -- they'd
   walk away from a screening thinking they're fine. A false positive just means an extra follow-up test for a
   healthy patient. For a screening tool, missing a real case is usually far more costly than a false alarm, so
   recall on the "disease" class tends to matter more than precision.
3. As `max_depth` increases, the tree is allowed to ask more and more specific questions, eventually carving out
   rules that fit quirks of the specific training rows rather than the general pattern -- that's why train
   accuracy keeps climbing toward a perfect score. Test accuracy only improves while the extra splits are
   capturing real signal; past some point the extra splits are memorizing training-set noise that doesn't
   generalize, so test accuracy stalls or drops even as train accuracy keeps rising.
4. Feature importance measures how useful a feature was for *this particular fitted tree's splits* -- not
   whether it's medically relevant in general. A feature can have low importance because a correlated feature
   (like `ST_Slope` or `ChestPainType`) already captures most of the same signal, so the tree never needed to
   split on `Cholesterol` separately. Low importance in one model is not the same claim as "no medical
   relevance."

</details>